# EDSS 전체 패널 키·내용 감사

## tl;dr

233개 패널·180,119,183행의 파일 및 행 추적 무결성에서 critical 오류는 발견되지 않았다. 다만 211개 패널은 `(연도, 개방ID)`보다 세분된 grain이 필요하고, 4개 패널은 `0101` 미연결률이 1%를 넘는다. 원시 `0101` 결합은 229개 패널에서 이론상 30,192,638개의 추가 행을 만들 수 있으므로 안전 결합 기준표를 사용해야 한다.

## Context & Methods

이 노트북은 `scripts/audit_edss_full_panel_keys.py`가 생성한 패널·연도·미연결 키 결과를 재검산하는 동반 산출물이다. 감사기는 모든 gzip 행을 읽어 SHA-256, 행 ID, 행 폭, 연도, `개방ID`, 후보키 반복, `0101` 연결과 원시 조인 증식 위험을 계산했다.

### Key Assumptions

- `_panel_year`와 `개방ID`를 공통 학교연도 후보키로 사용한다.
- `0101 고등교육학교개황`은 연결 범위의 기준이지 유일한 학교 마스터라고 가정하지 않는다.
- source row hash는 원본 멤버의 당시 열 순서를 사용하므로 합집합 열 패널만으로 재구성되지 않는 행을 손상으로 판정하지 않는다.
- 추가 차원 순위는 최대 10,000행의 결정적 표본이며 최종 키 확정 검사가 아니다.

## Data

### 1. 결과 파일 로드

In [1]:
from pathlib import Path
import csv
import json
from collections import Counter

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'data/metadata/edss_full_panel_key_audit.json').exists()
)
summary_path = repo_root / 'data/metadata/edss_full_panel_key_audit.json'
panels_path = repo_root / 'data/metadata/edss_full_panel_key_audit.csv'
orphans_path = repo_root / 'data/metadata/edss_full_panel_orphan_school_year_keys.csv'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
with panels_path.open(encoding='utf-8-sig', newline='') as handle:
    panels = list(csv.DictReader(handle))
with orphans_path.open(encoding='utf-8-sig', newline='') as handle:
    orphans = list(csv.DictReader(handle))

{'panels': len(panels), 'rows': summary['total_rows'], 'orphan_keys': len(orphans), 'status': summary['status']}

{'panels': 233,
 'rows': 180119183,
 'orphan_keys': 3322,
 'status': 'review_required'}

### 2. 핵심 합계 재검산

In [2]:
assert summary['audit_version'] == '2'
assert len(panels) == summary['logical_panel_count'] == 233
assert sum(int(row['row_count']) for row in panels) == summary['total_rows'] == 180_119_183
assert len(orphans) == summary['join_integrity']['orphan_school_year_keys'] == 3_322
assert sum(int(row['row_count']) for row in orphans) == summary['join_integrity']['orphan_rows'] == 82_959
assert summary['integrity']['width_mismatch_rows'] == 0
assert summary['integrity']['row_id_mismatches'] == 0
assert summary['integrity']['duplicate_row_ids'] == 0
assert summary['integrity']['exact_canonical_row_duplicates'] == 0
'All aggregate checks passed.'

'All aggregate checks passed.'

## Results

### 3. 심각도와 grain

In [3]:
{
    'severity': summary['severity_counts'],
    'grain': summary['grain_status_counts'],
    'open_id': summary['open_id'],
}

{'severity': {'high': 4, 'medium': 229},
 'grain': {'additional_dimensions_required': 211,
  'partial_id_coverage': 1,
  'school_year_open_id_unique': 21},
 'open_id': {'panels_with_column': 233,
  'panels_with_missing_rows': 1,
  'missing_rows': 46962,
  'whitespace_rows': 0,
  'normalization_collisions': 0}}

### 4. High 패널

In [4]:
high_panels = [
    {
        'source': row['source'],
        'code': row['catalog_code'],
        'dataset': row['dataset'],
        'orphan_rows': int(row['orphan_row_count']),
        'orphan_rate_pct': round(float(row['orphan_row_rate']) * 100, 4),
    }
    for row in panels if row['severity'] == 'high'
]
assert len(high_panels) == 4
high_panels

[{'source': '대학정보공시',
  'code': '0202',
  'dataset': '대학입학전형기본계획_전문대학',
  'orphan_rows': 218,
  'orphan_rate_pct': 1.3373},
 {'source': '대학정보공시',
  'code': '0204',
  'dataset': '대학입학전형시행계획_전문대학',
  'orphan_rows': 136,
  'orphan_rate_pct': 1.3053},
 {'source': '대학정보공시',
  'code': '1102',
  'dataset': '도서관예산현황',
  'orphan_rows': 35,
  'orphan_rate_pct': 1.5466},
 {'source': '대학정보공시',
  'code': '1209',
  'dataset': '법인임원현황',
  'orphan_rows': 788,
  'orphan_rate_pct': 1.064}]

### 5. 미연결 키의 시간적 위치

In [5]:
orphan_keys_by_class = Counter(row['classification'] for row in orphans)
orphan_rows_by_class = Counter()
for row in orphans:
    orphan_rows_by_class[row['classification']] += int(row['row_count'])
{'keys': dict(orphan_keys_by_class), 'rows': dict(orphan_rows_by_class)}

{'keys': {'after_last_0101_year': 1341,
  'before_first_0101_year': 1668,
  'internal_0101_gap': 217,
  'never_in_0101': 96},
 'rows': {'after_last_0101_year': 54614,
  'before_first_0101_year': 18198,
  'internal_0101_gap': 9033,
  'never_in_0101': 1114}}

### 6. 원시 `0101` 조인 위험

In [6]:
reference = next(row for row in panels if row['source'] == '고등교육통계' and row['catalog_code'] == '0101')
{
    '0101_school_year_keys': int(reference['school_year_key_count']),
    '0101_repeated_keys': int(reference['repeated_school_year_key_count']),
    '0101_max_multiplicity': int(reference['max_school_year_key_multiplicity']),
    'affected_panels': summary['join_integrity']['panels_with_join_expansion_risk'],
    'affected_rows': summary['join_integrity']['join_expansion_affected_rows'],
    'theoretical_extra_rows': summary['join_integrity']['join_expansion_extra_rows'],
}

{'0101_school_year_keys': 30556,
 '0101_repeated_keys': 1187,
 '0101_max_multiplicity': 4,
 'affected_panels': 229,
 'affected_rows': 27047504,
 'theoretical_extra_rows': 30192638}

## Takeaways

- 파일·행 추적 무결성 검사에는 critical 오류가 없다.
- `(연도, 개방ID)`는 대부분의 패널에서 완전한 행 키가 아니며, 패널별 추가 차원을 확정해야 한다.
- `0101` 원시 결합은 행 증식을 일으키므로 `data/metadata/edss_school_year_bridge.csv`를 사용한다.
- 3,322개 미연결 키는 삭제하거나 추정 매핑하지 말고 학교 이력·조사 범위와 교차검증한다.
- 취업통계 2023–2024년의 `개방ID` 결측 46,962행은 별도 학교명 교차표 없이는 장기 학교 패널에 직접 연결하지 않는다.